In [ ]:
# Import Pandas
import pandas as pd

#Load Data
data = pd.read_csv('/content/tweets.csv')
data

In [ ]:
data.info() # Data info

In [ ]:
df=data.drop("id",axis=1) #drop the id column

In [ ]:
# Import pandas,train_test_split,CountVectorizer,MultinomialNB,accuracy_score, classification_report, confusion_matrix
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import pandas as pd
import re # for replacing
import nltk # natural language tool kit
from nltk.corpus import stopwords # Importing stopwords

nltk.download('stopwords') # downloading stop words
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower() # converting to Lowercase
    text = re.sub(r'http\S+|@\w+|#\w+', '', text) # Removing http,@,# char and replace it with ''
    text = re.sub(r'[^a-z\s]', '', text) # only keep letters and replace all others with ''
    words = text.split() # splitting the words
    words = [w for w in words if w not in stop_words] # will not include stopwords
    return " ".join(words)

df['clean_text'] = df['tweet'].apply(clean_text)

In [ ]:
#df['label'] = df['label'].replace(1, 'Negative') # Replacing label 1 with Negative
#df['label'] = df['label'].replace(0, 'Positive') # Replacing label  with Negative
df["label"].value_counts()

In [ ]:
df

In [ ]:
# Import CountVectorizer and  numpy
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['clean_text']) # transforming the cleaned data

co_matrix = (X.T @ X) # association btw the words
co_matrix.setdiag(0)

words = vectorizer.get_feature_names_out() # get feature names
co_df = pd.DataFrame(co_matrix.toarray(), index=words, columns=words)

print(co_df.head())

In [ ]:
print(co_df['angry'].sort_values(ascending=False).head(10)) # finding similar words to angry

In [ ]:
print(co_df['happy'].sort_values(ascending=False).head(10)) # finding similar words to happy

In [ ]:
# importing TfidfVectorizer,train_test_split,LogisticRegression,SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE


tfidf = TfidfVectorizer(max_features=5000) # vectorizer is used to convert words to numbers

X = tfidf.fit_transform(df['clean_text'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # Splitting the data to train and test
smote = SMOTE(sampling_strategy='minority', random_state=42)# Appling smote
X_train, y_train= smote.fit_resample(X_train, y_train)
y_train.value_counts()

model = LogisticRegression(max_iter=200) # Logistic Regression model
model.fit(X_train, y_train ) # Training the model

print(model.score(X_test, y_test))

In [ ]:
import nltk #library # natural language tool kit
import numpy as np
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet #stopwords
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('stopwords')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
mapping = {0: "positive", 1: "negative"}


new_text = "I am so happy today"


text = new_text.lower()  # 1. Lowercase
text = re.sub(r'[^a-z\s]', '', text)  # 2. Remove punctuation/numbers
words = text.split()     # 3. Tokenize
print(words)
words = [w for w in words if w not in stop_words]  # 4. Remove stopwords
words = [lemmatizer.lemmatize(w,pos="v") for w in words]   # 5. Lemmatization

# transform using SAME tfidf
new_vec = tfidf.transform(words)

# predict
prediction = model.predict(new_vec)
import numpy as np
pred_class = np.argmax(prediction)

print("Predicted Class:", pred_class)
#label = le.inverse_transform([pred_class])
label = ["positive", "negative"]
print(label[pred_class])



In [ ]:
# BART
from transformers import pipeline # Imported pipeline

classifier = pipeline(
    "text-classification", # To Text Classification
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print(classifier("I don't Like this Laptop!"))

In [ ]:
# Tokenization + Padding
from tensorflow.keras.preprocessing.text import Tokenizer #imported Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences #imported pad_Sequence

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_text'])

seqs = tokenizer.texts_to_sequences(df['clean_text']) # Text to seq
X_pad = pad_sequences(seqs, maxlen=100) # maxlength for each tweet

In [ ]:
# 1.RNN
from tensorflow.keras.models import Sequential # imported Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense # imported Embedding, SimpleRNN, Dense

model = Sequential([
    Embedding(5000, 128),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=100)

In [ ]:
#from sklearn.preprocessing import LabelEncoder # imported  LabelEncoder

#le = LabelEncoder()
y = df['label'] # Encoded the Y

print(y[:5])

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=5)

In [ ]:
new_text = ["I am happy today."]


# step 1: tokenize
seq = tokenizer.texts_to_sequences(new_text)


# step 2: pad
padded = pad_sequences(seq, maxlen=100)

# step 3: predict
pred = model.predict(padded)

print("Predicted Class :",pred)
#label = le.inverse_transform([pred_class])
print(label[pred_class])


In [ ]:
print(np.unique(y_train, return_counts=True))

In [ ]:
import numpy as np

pred_class = np.argmax(pred)
print(pred_class)

In [ ]:
label = label[pred_class]
print("Predicted Class :",label)

In [ ]:
#2.lstm

from tensorflow.keras.layers import LSTM

model = Sequential([
    Embedding(5000, 128),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test data
    random_state=42     # reproducibility
)
smote = SMOTE(sampling_strategy='minority', random_state=42)# Appling smote
X_train, y_train= smote.fit_resample(X_train, y_train)


In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [ ]:

new_text = ["I feel so happy and excited"]

seq = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(seq, maxlen=100)

pred = model.predict(padded)


pred_class = int(pred[0][0] > 0.5)

print("Predicted Class:", pred_class)

predicted_label = label[pred_class]
print("Predicted Label:", predicted_label)

In [ ]:
#  GRU

from tensorflow.keras.layers import GRU

model = Sequential([
    Embedding(5000, 128),
    GRU(64),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [ ]:
new_text = ["i am so happy today"]

seq = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(seq, maxlen=100)

pred = model.predict(padded)


pred_class = int(pred[0][0] > 0.5)

print("Predicted Class:", pred_class)

predicted_label = labels[pred_class]
print("Predicted Label:", predicted_label)
